# argmax(l1, l2) with l2 = gemini + os_distill — per-lane relabel of the unreached rows

The follow-up `argmax_enrichment_pilot.ipynb` named: the same select-the-more-decisive-leg
mechanism, but l2's sparse side upgraded from bm25 to **os_distill** (bake-off winner,
sparse-only coverage 0.392 → 0.650, [[project-leg2-sparse-bakeoff]]), run over the
**v2-100K unreached population** instead of the arch5k draw.

**Per lane:**
1. take its `all_zero` / `all_tied` rows from `rungs/100k-v2/labeling/labels.parquet`
   (natural-scored only — supplemented rows are scored against constructed docs the
   distill collections don't hold, the same confound `LegBPilot._selection` drops);
2. relabel them through the lane's `..._gemini-embedding-001_opensearch-...-distill_routes`
   collection (l2 triples, written incrementally to `data/legb_pilot/os_distill_relabel/`);
3. combine: per row keep the **whole triple of the more decisive leg** (`select_leg` —
   never mix routes across legs) and count how many rows the lane moved off zero/tie
   into a decisive label.

**Scope**: only lanes whose distill collection already exists on the cluster (12 as of
09-06 — the banked ~94% of unreached rows). Lanes without one are listed and skipped,
never indexed: indexing is `apply_os_distill_sparse.py`'s job and costs dense spend.

**Cost**: query-side only. Gemini embeds each query text server-side (dense_only +
pure_rrf calls), os_distill's query side is inference-free (tokenizer + IDF lookup).
The setup cell prints the estimate; expect well under $1 for ~40K short queries.
Wall-clock is the real budget — three Qdrant searches per row.

**Also read out**: the winner-route mix of the rescued rows — arch5k finding 4 was an
87%-dense skew with l2 = gemini+bm25; [[project-judge-roadmap]] holds residual scaling
until this is re-checked with the os_distill leg. This table is that re-check.

In [9]:
# Setup — all free reads. RUN_LIVE in §2 is the only spend point.
from __future__ import annotations

import os
import pathlib
import sys

SRC = pathlib.Path.cwd()
SRC = next(parent / "src" for parent in (SRC, *SRC.parents)
    if (parent / "src" / "hybrid_search_rrf_dataset").is_dir())
sys.path.insert(0, str(SRC))

import pandas as pd
from dotenv import load_dotenv

from relevance_judge.config import RelevanceJudgeConfig

load_dotenv()
pd.set_option("display.width", 170)

CONFIG = RelevanceJudgeConfig()
OUT_DIR = SRC / "data" / "legb_pilot" / "os_distill_relabel"
UNREACHED = ("all_zero", "all_tied")
ROUTES = ("dense_only", "pure_rrf", "sparse_only")
SCORE_COLS = [f"score_{r}" for r in ROUTES]

L1 = pd.read_parquet(CONFIG.labels).astype({"query_id": str})
# Blank queries can't be routed and the cloud embed 400s on them — their l1
# all_zero is a no-query artifact, not signal (13 such rows, all crumb-code).
blank = L1["query"].fillna("").str.strip() == ""
SEL = L1[L1["shape"].isin(UNREACHED) & (L1["scored_against"] == "natural") & ~blank]
if blank.any():
    print(f"dropped {blank.sum()} blank-query rows: "
          f"{dict(L1.loc[blank, 'dataset'].value_counts())}")

q_mtok = SEL["query"].str.len().sum() / 4 / 1e6
print(f"v2-100K rows: {len(L1):,}   unreached: {L1['shape'].isin(UNREACHED).sum():,}"
      f"   natural-scored unreached (this notebook's population): {len(SEL):,}")
print(f"gemini query-side estimate: ~{q_mtok:.1f}M tok x 2 embeds x $0.15/M"
      f" = ~${q_mtok * 2 * 0.15:.2f}")
SEL.groupby("dataset")["shape"].value_counts().unstack(fill_value=0)

dropped 13 blank-query rows: {'crumb-code-retrieval': np.int64(13)}
v2-100K rows: 91,093   unreached: 48,222   natural-scored unreached (this notebook's population): 39,986
gemini query-side estimate: ~2.1M tok x 2 embeds x $0.15/M = ~$0.63


shape,all_tied,all_zero
dataset,,
antique,1,10
beir-nfcorpus,17,23
beir-touche-2020,2,0
bright-aops,0,48
bright-biology,4,23
bright-earth-science,9,17
bright-economics,6,41
bright-leetcode,19,49
bright-pony,0,26


## 1 · Which lanes are runnable — the banked distill collections

Read-only Qdrant listing. A lane runs only if its
`<source>_legb_gemini-embedding-001_opensearch-neural-sparse-encoding-doc-v3-distill_routes`
collection exists AND holds its full corpus (`indexable` docs) — a partial collection
would silently score l2 against a truncated corpus and manufacture all_zero rows.

In [10]:
from qdrant_client import QdrantClient

from hybrid_search_rrf_dataset.retrieval import SnapshotDataset
from scripts.apply_os_distill_sparse import _pilot
from scripts.label_routes import DATA_DIR, _source_name
from scripts.legb import indexable

client = QdrantClient(url=os.environ["QDRANT_CLOUD_URL"],
                      api_key=os.environ["QDRANT_CLOUD_API_KEY"],
                      timeout=120, cloud_inference=True)

LANE_ORDER = SEL["dataset"].value_counts().index.tolist()  # biggest payoff first
pilot = _pilot(client, tuple(LANE_ORDER))
live = {c.name for c in client.get_collections().collections}

runnable, skipped = [], []
for lane in LANE_ORDER:
    coll = pilot.collection(lane)
    if coll not in live:
        skipped.append(lane)
        continue
    got = client.count(coll, exact=True).count
    want = indexable(SnapshotDataset(_source_name(lane), path=str(DATA_DIR)).corpus())
    runnable.append({"lane": lane, "unreached": int((SEL["dataset"] == lane).sum()),
                     "points": got, "corpus": want, "complete": got >= want})

plan = pd.DataFrame(runnable)
# Small collections first: search latency scales with collection size (measured
# 0.3s/call on 10K docs vs 13-20s on 120K under load), so this banks the cheap
# lanes before the big grinds — an abort loses grind, not breadth. Finished
# lanes skip via the §2 pre-filter regardless of order.
LANES_TO_RUN = plan.loc[plan["complete"]].sort_values("corpus")["lane"].tolist()
print(f"runnable: {len(LANES_TO_RUN)} lanes, {plan.loc[plan['complete'], 'unreached'].sum():,} rows"
      f"  ({plan.loc[plan['complete'], 'unreached'].sum() / len(SEL):.0%} of the population)")
print(f"run order (small collections first): {LANES_TO_RUN}")
print(f"no distill collection (not touched here): {skipped}")
plan

runnable: 12 lanes, 37,156 rows  (93% of the population)
run order (small collections first): ['techqa', 'scirgen-geo-en', 'finder', 'crumb-legal-qa', 'clerc', 'rarb-math', 'gooaq', 'webfaq-eng', 'orcas', 'quest', 'msmarco-passage-dev', 'crumb-code-retrieval']
no distill collection (not touched here): ['miracl-en-dev', 'lotte-technology-forum', 'lotte-technology-search', 'bright-theoremqa-questions', 'wands', 'crumb-set-operation-entity-retrieval', 'limit', 'bright-leetcode', 'crumb-tip-of-the-tongue', 'bright-theoremqa-theorems', 'crumb-stack-exchange', 'crumb-theorem-retrieval', 'bright-aops', 'bright-economics', 'bright-robotics', 'bright-psychology', 'beir-nfcorpus', 'freshstack-godot', 'bright-stackoverflow', 'freshstack-laravel', 'bright-sustainable-living', 'freshstack-langchain', 'bright-biology', 'dbpedia-entity', 'bright-earth-science', 'bright-pony', 'freshstack-yolo', 'freshstack-angular', 'antique', 'trec-dl-2022', 'beir-touche-2020', 'crumb-clinical-trial']


,lane,unreached,points,corpus,complete
0,scirgen-geo-en,13513,3349,3349,True
1,webfaq-eng,4605,43060,43060,True
2,finder,3690,5830,5830,True
3,rarb-math,3248,31595,31595,True
4,gooaq,2981,35375,35375,True
5,crumb-code-retrieval,2891,119976,119976,True
6,crumb-legal-qa,2637,10000,10000,True
7,orcas,1176,62239,62239,True
8,msmarco-passage-dev,795,100000,100000,True
9,rarb-code,620,8000,100000,False


## 2 · Relabel each lane through its l2 stack — `RUN_LIVE`-gated

`RouteLabels.label` is incremental: already-labelled `query_id`s in
`os_distill_relabel/labels.parquet` are skipped, so a killed run resumes where it
stopped and re-running a finished lane costs nothing. `min_relevance` comes from
`LANES`, matching what l1 used — the two legs must read the same gold or the diff
measures the objective, not the encoder.

In [11]:
RUN_LIVE = False  # flip to spend: gemini query embeds + hours of Qdrant searches

# Rows are network-latency-bound, not compute-bound (measured: ~700ms per
# dense/rrf call — the server-side gemini embed hop — 330ms sparse, 2ms local
# encode). Workers exist to hide that latency; the ceiling is provider rate
# limits, not this machine.
WORKERS = 8

# l2 labels land in their OWN artifact — OUT_DIR/labels.parquet — a fresh dataset.
# The v2-100K labeling file this notebook reads (CONFIG.labels) is never written.
L2_PATH = OUT_DIR / "labels.parquet"
assert L2_PATH.resolve() != CONFIG.labels.resolve()

if RUN_LIVE:
    import time
    from datetime import datetime, timedelta

    from tqdm.auto import tqdm

    from hybrid_search_rrf_dataset.fusion import (
        DenseOnlyStrategy, PureRRFStrategy, SparseOnlyStrategy,
    )
    from hybrid_search_rrf_dataset.labels import RouteLabels
    from hybrid_search_rrf_dataset.lanes import LANES
    from hybrid_search_rrf_dataset.objective import RouterObjective
    from scripts.legb import gemini_dense_cfg, opensearch_distill_sparse_cfg

    dense, sparse = gemini_dense_cfg(), opensearch_distill_sparse_cfg()
    # rows already banked in the l2 artifact are never rescored — label()
    # enforces this; the pre-filter makes the skip visible and jumps lanes
    # that are already finished without touching Qdrant or the query embeds
    done = (pd.read_parquet(L2_PATH, columns=["dataset", "query_id"]).astype(str)
            if L2_PATH.exists() else pd.DataFrame(columns=["dataset", "query_id"]))

    # the whole run's work list up front, so progress runs against a fixed total
    todo = {}
    for lane in LANES_TO_RUN:
        sel = SEL.loc[SEL["dataset"] == lane, ["dataset", "query_id", "query"]].astype(str)
        banked = done.loc[done["dataset"] == lane, "query_id"]
        todo[lane] = sel[~sel["query_id"].isin(banked)]
    total = sum(len(s) for s in todo.values())
    print(f"run plan: {total:,} rows across {sum(bool(len(s)) for s in todo.values())} lanes"
          f" ({len(done):,} rows already banked). Intra-lane progress is label()'s own"
          f" chunk bar — one chunk = 500 rows = one parquet checkpoint.")

    run_t0 = time.monotonic()
    results = []
    overall = tqdm(total=total, desc="all lanes", unit="row", smoothing=0)
    for i, lane in enumerate(LANES_TO_RUN, 1):
        sel = todo[lane]
        if sel.empty:
            tqdm.write(f"=== [{i}/{len(LANES_TO_RUN)}] {lane} — already complete, skipped ===")
            results.append({"lane": lane, "rows": 0, "min": 0.0, "rows_per_s": None,
                            "status": "already banked"})
            continue
        overall.set_postfix_str(lane)
        tqdm.write(f"=== [{i}/{len(LANES_TO_RUN)}] {lane} — {len(sel):,} to label ===")
        min_rel = LANES[lane].min_relevance if lane in LANES else 1
        labels = RouteLabels(sel, out_dir=OUT_DIR,
                             objective=RouterObjective(min_relevance=min_rel))
        coll = pilot.collection(lane)
        args = (client, coll, dense, sparse)
        source = SnapshotDataset(_source_name(lane), path=str(DATA_DIR))
        t0 = time.monotonic()
        try:
            out = labels.label(source,
                               DenseOnlyStrategy(*args), PureRRFStrategy(*args),
                               SparseOnlyStrategy(*args),
                               dataset=lane, max_workers=WORKERS)
        except ValueError as error:  # "no rows produced" — record it, keep the run alive
            tqdm.write(f"[{lane}] FAILED: {error}")
            results.append({"lane": lane, "rows": 0,
                            "min": round((time.monotonic() - t0) / 60, 1),
                            "rows_per_s": None, "status": f"failed: {error}"})
            continue
        dt = time.monotonic() - t0
        overall.update(len(out))
        shape = out["shape"].value_counts()
        # run-level ETA from measured row throughput, not lane count — lanes
        # are wildly uneven (scirgen alone is ~36% of all rows)
        rate = overall.n / (time.monotonic() - run_t0)
        eta = datetime.now() + timedelta(seconds=(total - overall.n) / max(rate, 1e-9))
        tqdm.write(
            f"[{lane}] {dt / 60:.1f} min at {len(out) / dt:.1f} rows/s — "
            f"+{len(out):,} labels  "
            f"differ {shape.get('routes_differ', 0):,} | "
            f"tied {shape.get('all_tied', 0):,} | zero {shape.get('all_zero', 0):,}   "
            f"[run: {overall.n:,}/{total:,}, eta {eta:%H:%M}]"
        )
        results.append({"lane": lane, "rows": len(out), "min": round(dt / 60, 1),
                        "rows_per_s": round(len(out) / dt, 1),
                        "differ": int(shape.get("routes_differ", 0)),
                        "tied": int(shape.get("all_tied", 0)),
                        "zero": int(shape.get("all_zero", 0)), "status": "ok"})
    overall.close()

    print(f"\n=== run summary — {(time.monotonic() - run_t0) / 60:.1f} min total ===")
    display(pd.DataFrame(results))
else:
    print("RUN_LIVE=False — reading whatever the last run banked in", OUT_DIR)

RUN_LIVE=False — reading whatever the last run banked in /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/legb_pilot/os_distill_relabel


## 3 · Combine — argmax(l1, l2) per row, improvement per lane

`select_leg` (same as the enrichment pilot): keep the WHOLE triple of the more
decisive leg — bigger top-vs-runner-up margin, then bigger top. Never mixes routes
across legs, so a combined tie means BOTH legs measured one. On this population l1's
triple is degenerate by construction (all-zero or all-tied), so any l2 movement wins
the selection — the table below is therefore exactly "what l2 adds".

- **rescued** — combined triple has a strict, non-zero winner: an unreached row
  became a decisive label.
- **still_tied / still_zero** — l2 didn't break it either (for all-tied rows at the
  qrels ceiling this is expected: [[project-tie-ceiling]] — no encoder can move a row
  where every route already ranks the judged doc first).

In [12]:
TOL = 1e-9


def triple(row) -> dict[str, float]:
    return {r: float(row[f"score_{r}"]) for r in ROUTES}


def winner(scores: dict) -> str | None:
    ordered = sorted(scores, key=scores.get, reverse=True)
    top = scores[ordered[0]]
    if top <= TOL or top - scores[ordered[1]] <= TOL:
        return None
    return ordered[0]


def select_leg(l1_scores: dict, l2_scores: dict) -> dict:
    margin = lambda s: (lambda a: a[0] - a[1])(sorted(s.values(), reverse=True))
    k1 = (margin(l1_scores), max(l1_scores.values()))
    k2 = (margin(l2_scores), max(l2_scores.values()))
    return l2_scores if k2 > k1 else l1_scores


L2 = pd.read_parquet(OUT_DIR / "labels.parquet").astype({"query_id": str})
merged = SEL.merge(L2, on=["dataset", "query_id"], suffixes=("_l1", "_l2"))
print(f"l2-labelled rows joined: {len(merged):,} of {len(SEL):,} selected")

rows = []
for _, r in merged.iterrows():
    t1 = {rt: float(r[f"score_{rt}_l1"]) for rt in ROUTES}
    t2 = {rt: float(r[f"score_{rt}_l2"]) for rt in ROUTES}
    comb = select_leg(t1, t2)
    w = winner(comb)
    rows.append({"dataset": r["dataset"], "l1_shape": r["shape_l1"],
                 "rescued": w is not None, "route": w,
                 "outcome": w or ("still_zero" if max(comb.values()) <= TOL else "still_tied")})
combined = pd.DataFrame(rows)

lane_table = (combined.groupby("dataset")
              .agg(unreached=("rescued", "size"), rescued=("rescued", "sum"))
              .assign(rescue_pct=lambda d: (d["rescued"] / d["unreached"] * 100).round(1))
              .sort_values("rescued", ascending=False))
lane_table.loc["TOTAL"] = [lane_table["unreached"].sum(), lane_table["rescued"].sum(),
                           round(lane_table["rescued"].sum() / lane_table["unreached"].sum() * 100, 1)]
lane_table.astype({"unreached": int, "rescued": int})

l2-labelled rows joined: 37,156 of 39,986 selected


,unreached,rescued,rescue_pct
dataset,,,
crumb-code-retrieval,2891,2609,90.2
scirgen-geo-en,13513,1371,10.1
crumb-legal-qa,2637,1244,47.2
finder,3690,1243,33.7
rarb-math,3248,604,18.6
quest,614,303,49.3
gooaq,2981,238,8.0
orcas,1176,202,17.2
webfaq-eng,4605,197,4.3


In [13]:
# Dataset-level ledger: the argmax verdict projected onto ALL of v2-100K.
# Only unreached (all_tied/all_zero), natural-scored rows in the 12 distill
# lanes were relabelled; every other row keeps its l1 shape — routes_differ
# was never in scope (l1 already decided it), supplemented rows are confounded
# by constructed docs, skipped lanes have no collection, blank queries are junk.
after_shape = combined["outcome"].replace(
    {"still_tied": "all_tied", "still_zero": "all_zero"}
).where(combined["outcome"].isin(["still_tied", "still_zero"]), "decisive")

SHAPES = ["routes_differ", "all_tied", "all_zero"]
before_full = L1["shape"].value_counts().reindex(SHAPES, fill_value=0)
moved = combined["l1_shape"].value_counts()          # leaves its l1 bucket
landed = after_shape.replace({"decisive": "routes_differ"}).value_counts()
after_full = before_full.sub(moved, fill_value=0).add(landed, fill_value=0) \
                        .reindex(SHAPES, fill_value=0).astype(int)

N = len(L1)
ledger = pd.DataFrame({
    "before (l1)": before_full,
    "before %": (before_full / N * 100).round(1),
    "after (argmax)": after_full,
    "after %": (after_full / N * 100).round(1),
    "delta": after_full - before_full,
    "delta pp": ((after_full - before_full) / N * 100).round(1),
})
ledger.loc["total"] = [N, 100.0, N, 100.0, 0, 0.0]
ledger = ledger.astype({"before (l1)": int, "after (argmax)": int, "delta": int})

n_new_decisive = int((after_shape == "decisive").sum())
untouched = N - len(combined)
supp = int((L1["shape"].isin(UNREACHED) & (L1["scored_against"] != "natural")).sum())
print(f"v2-100K: {N:,} rows | relabelled {len(combined):,} | untouched {untouched:,} "
      f"(= {int(before_full['routes_differ']):,} already-decided routes_differ "
      f"+ {supp:,} supplemented-scored unreached + {untouched - int(before_full['routes_differ']) - supp:,} "
      f"no-collection/blank)")
print(f"of the after routes_differ, {n_new_decisive:,} are NEW decisive labels from the argmax\n")
display(ledger)

print("transition matrix over the relabelled slice (rows = l1 shape, cols = after argmax):")
display(pd.crosstab(combined["l1_shape"], after_shape, margins=True, margins_name="total"))

v2-100K: 91,093 rows | relabelled 37,156 | untouched 53,937 (= 42,871 already-decided routes_differ + 8,223 supplemented-scored unreached + 2,843 no-collection/blank)
of the after routes_differ, 8,132 are NEW decisive labels from the argmax



,before (l1),before %,after (argmax),after %,delta,delta pp
shape,,,,,,
routes_differ,42871,47.1,51003,56.0,8132,8.9
all_tied,24751,27.2,25350,27.8,599,0.7
all_zero,23471,25.8,14740,16.2,-8731,-9.6
total,91093,100.0,91093,100.0,0,0.0


transition matrix over the relabelled slice (rows = l1 shape, cols = after argmax):


outcome,all_tied,all_zero,decisive,total
l1_shape,,,,
all_tied,15899,0,754,16653
all_zero,1353,11772,7378,20503
total,17252,11772,8132,37156


In [14]:
# Rescue split by what l1 said (zero vs tie), and the dense-skew re-check:
# with l2 = gemini+bm25 the arch5k winner mix was 87% dense (finding 4).
print("rescue rate by l1 shape:")
print(combined.groupby("l1_shape")["rescued"].agg(["size", "sum", "mean"])
      .rename(columns={"size": "rows", "sum": "rescued", "mean": "rate"}).to_string())

won = combined[combined["rescued"]]
print(f"\nwinner-route mix of the {len(won):,} rescued rows"
      " (the 87%-dense skew re-check, [[project-judge-roadmap]]):")
print((won["route"].value_counts(normalize=True) * 100).round(1).to_string())
print("\nper lane:")
won.groupby("dataset")["route"].value_counts().unstack(fill_value=0)

rescue rate by l1 shape:
           rows  rescued      rate
l1_shape                          
all_tied  16653      754  0.045277
all_zero  20503     7378  0.359850

winner-route mix of the 8,132 rescued rows (the 87%-dense skew re-check, [[project-judge-roadmap]]):
route
dense_only     86.8
sparse_only    11.3
pure_rrf        1.9

per lane:


route,dense_only,pure_rrf,sparse_only
dataset,,,
clerc,24,5,28
crumb-code-retrieval,2564,23,22
crumb-legal-qa,1088,57,99
finder,1166,14,63
gooaq,176,2,60
msmarco-passage-dev,20,3,26
orcas,145,4,53
quest,249,15,39
rarb-math,601,0,3


## 5 · Depth-100 fallback — NDCG@100 argmax over the surviving zeros

The deep-rank probe on the survivors: gold is a single doc, present in the
collection, ranked 11–100 by the best leg for ~45–50% of them — the top-10
window is simply narrower than depth-1 gold can satisfy on homogeneous
corpora. The fallback: for rows still all_zero under BOTH legs, rescore at
depth 100 (pure NDCG@100 — the main objective's terms are all zero here, so
a composite's 0.1 coefficient would be cosmetic) and argmax that.

Fallback-layer semantics: existing labels are never touched — this only adds
labels where there were none. Recovered rows carry `label_layer =
"depth100_fallback"`: gold-at-rank-23-vs-71 is genuine route-ordering signal
but weaker supervision than a top-10 hit (tiny margins, partly arbitrary on
scirgen's near-duplicate corpus) — downstream selection decides their weight,
and a judge eye-test on a sample is due before they enter supply
(the arch5k rank-instrument precedent earns them that much skepticism).

In [15]:
RUN_DEPTH100 = False  # flip to spend: ~35K searches at depth 100
D100_WORKERS = 16    # network-bound, same as the main run; drop to 8 if 500-retries appear
CHECKPOINT = 500     # rows per parquet write — a killed run loses at most this many

import math
from concurrent.futures import ThreadPoolExecutor, as_completed

DEPTH = 100
D100_PATH = OUT_DIR / "depth100_rescue.parquet"

# the fallback population: rows all_zero under BOTH legs (l1 and the l2 relabel)
zero_l1 = SEL[SEL["shape"] == "all_zero"][["dataset", "query_id", "query"]].astype(str)
zero_l2 = L2.loc[L2["shape"] == "all_zero", ["dataset", "query_id"]].astype(str)
survivors = zero_l1.merge(zero_l2, on=["dataset", "query_id"])
print(f"surviving zeros to rescore at depth {DEPTH}: {len(survivors):,}")


def ndcg_at(ranking_ids, gold: set, depth: int) -> float:
    dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(ranking_ids[:depth]) if d in gold)
    ideal = sum(1 / math.log2(i + 2) for i in range(min(len(gold), depth)))
    return dcg / ideal if ideal else 0.0


if RUN_DEPTH100:
    from hybrid_search_rrf_dataset.fusion import (
        DenseOnlyStrategy, PureRRFStrategy, SparseOnlyStrategy,
    )
    from hybrid_search_rrf_dataset.lanes import LANES
    from scripts.legb import gemini_dense_cfg, opensearch_distill_sparse_cfg
    from tqdm.auto import tqdm

    done_d100 = (pd.read_parquet(D100_PATH).astype({"query_id": str})
                 if D100_PATH.exists() else pd.DataFrame(columns=["dataset", "query_id"]))
    banked_keys = set(map(tuple, done_d100[["dataset", "query_id"]].values))
    dense_cfg, sparse_cfg = gemini_dense_cfg(), opensearch_distill_sparse_cfg()
    out_rows: list[dict] = []

    def _save() -> None:
        pd.concat([done_d100, pd.DataFrame(out_rows)],
                  ignore_index=True).to_parquet(D100_PATH, index=False)

    for lane, grp in survivors.groupby("dataset"):
        todo_rows = grp[[k not in banked_keys for k in zip(grp["dataset"], grp["query_id"])]]
        if todo_rows.empty:
            continue
        min_rel = LANES[lane].min_relevance if lane in LANES else 1
        qr = pd.read_parquet(DATA_DIR / _source_name(lane) / "qrels.parquet").astype(
            {"query_id": str, "doc_id": str})
        gold_map = qr[qr["relevance"] >= min_rel].groupby("query_id")["doc_id"].apply(set)
        coll = pilot.collection(lane)
        strats = {
            "dense_only": DenseOnlyStrategy(client, coll, dense_cfg, sparse_cfg, fetch_limit=DEPTH),
            "pure_rrf": PureRRFStrategy(client, coll, dense_cfg, sparse_cfg, fetch_limit=DEPTH),
            "sparse_only": SparseOnlyStrategy(client, coll, dense_cfg, sparse_cfg, fetch_limit=DEPTH),
        }

        def score_row(row) -> dict:
            gold = gold_map.get(row.query_id, set())
            return {"dataset": lane, "query_id": row.query_id,
                    **{name: ndcg_at(list(s.rank(row.query)), gold, DEPTH)
                       for name, s in strats.items()}}

        with ThreadPoolExecutor(max_workers=D100_WORKERS) as pool:
            futures = [pool.submit(score_row, row)
                       for row in todo_rows.itertuples(index=False)]
            for f in tqdm(as_completed(futures), total=len(futures),
                          desc=f"d100:{lane}", unit="row"):
                out_rows.append(f.result())
                if len(out_rows) % CHECKPOINT == 0:
                    _save()
        _save()  # lane end

if D100_PATH.exists():
    d100 = pd.read_parquet(D100_PATH).astype({"query_id": str})
    d100["route"] = [winner({r: float(row[r]) for r in ROUTES}) for _, row in d100.iterrows()]
    d100["label_layer"] = "depth100_fallback"
    rec = d100[d100["route"].notna()]
    print(f"\ndepth-100 fallback over {len(d100):,} surviving zeros: "
          f"{len(rec):,} recovered ({len(rec) / max(len(d100), 1):.1%})")
    print("\nrecovered per lane:")
    print(rec["dataset"].value_counts().to_string())
    print("\nwinner-route mix of recovered rows:")
    print((rec["route"].value_counts(normalize=True) * 100).round(1).to_string())
else:
    print("no depth-100 pass banked yet — flip RUN_DEPTH100 to run it")

surviving zeros to rescore at depth 100: 11,772

depth-100 fallback over 11,772 surviving zeros: 5,044 recovered (42.8%)

recovered per lane:
dataset
scirgen-geo-en          3181
finder                   988
crumb-legal-qa           326
quest                    190
crumb-code-retrieval     123
gooaq                    113
orcas                     55
webfaq-eng                29
rarb-math                 21
clerc                      7
msmarco-passage-dev        6
techqa                     5

winner-route mix of recovered rows:
route
dense_only     66.0
sparse_only    31.7
pure_rrf        2.3


## 6 · The cascade, assembled — and the run's final record

Per unreached row, first layer with a decisive verdict wins:

1. **argmax(l1, l2)** at the main objective (0.7·HR@1 + 0.3·NDCG@10) — decisive → take its scores;
2. **depth-100 fallback** — NDCG@100 argmax over both-legs-zero rows — decisive → take its scores;
3. **relevance judge** — the residue's layer, not built yet: the remaining rows ARE its work-list.

`cascade_labels.parquet` records every measured row: `label_layer` says which
layer decided (or `judge_queue_*` for the residue), `route` the winner, and the
three scores from the deciding layer. **Scores are not comparable across
layers** (layer 1 is the serving objective, layer 2 is NDCG@100) — `label_layer`
is the column that keeps that honest. The 42,871 l1 routes_differ rows keep
their original record in the v2-100K labels.parquet; this artifact covers the
unreached population the cascade re-measured.

Each row also carries the **component scores** so a consumer can trace back and pick
which to trust without regenerating: `score_{route}__l1` (v2-100K serving objective),
`score_{route}__argmax` (argmax(l1,l2)), `score_{route}__d100` (top@100 NDCG, NaN where
not run), beside the unsuffixed `score_{route}` **final** (the deciding layer). To drop the
cross-objective top@100 noise, train on `score__argmax.fillna(score__l1)` — its both-legs-
zero rows revert to `all_zero` (kept, untrained).

In [16]:
CASCADE_PATH = OUT_DIR / "cascade_labels.parquet"

d100_map = (d100.set_index(["dataset", "query_id"])[[*ROUTES, "route"]]
            if D100_PATH.exists() else pd.DataFrame())

# --- the re-measured unreached rows: cascade layers 1/2/3 ---
records = []
for _, r in merged.iterrows():
    key = (r["dataset"], r["query_id"])
    t1 = {rt: float(r[f"score_{rt}_l1"]) for rt in ROUTES}
    t2 = {rt: float(r[f"score_{rt}_l2"]) for rt in ROUTES}
    comb = select_leg(t1, t2)
    w = winner(comb)
    if w is not None:                                    # layer 1: argmax(l1, l2)
        layer, route, scores = "argmax_l1_l2", w, comb
    elif key in d100_map.index and pd.notna(d100_map.loc[key, "route"]):
        row = d100_map.loc[key]                          # layer 2: depth-100 fallback
        layer, route = "depth100_fallback", row["route"]
        scores = {rt: float(row[rt]) for rt in ROUTES}
    else:                                                # layer 3: the judge's work-list
        still_zero = max(comb.values()) <= TOL
        layer = "judge_queue_zero" if still_zero else "judge_queue_tied"
        route, scores = None, comb
    d100_scores = ({rt: float(d100_map.loc[key, rt]) for rt in ROUTES}
                   if key in d100_map.index and pd.notna(d100_map.loc[key, "route"])
                   else {rt: float("nan") for rt in ROUTES})
    records.append({"dataset": r["dataset"], "query_id": r["query_id"],
                    "query": r["query_l1"],  # labels.parquet carries text for ALL
                    # provenances — queries.parquet is natural-only, never join it
                    "label_layer": layer, "route": route,
                    # final = the deciding layer; the component scores let a consumer
                    # trace back and pick l1 / argmax(l1,l2) / top@100 instead
                    **{f"score_{rt}": scores[rt] for rt in ROUTES},
                    **{f"score_{rt}__l1": t1[rt] for rt in ROUTES},
                    **{f"score_{rt}__argmax": comb[rt] for rt in ROUTES},
                    **{f"score_{rt}__d100": d100_scores[rt] for rt in ROUTES}})
cascade = pd.DataFrame(records)

# --- the rest of v2-100K, so the artifact covers ALL 91K rows for training ---
in_cascade = set(zip(cascade["dataset"], cascade["query_id"]))
rest = L1[[not k in in_cascade for k in zip(L1["dataset"], L1["query_id"].astype(str))]].copy()
rest["query_id"] = rest["query_id"].astype(str)
differ = rest["shape"] == "routes_differ"
rest_out = pd.DataFrame({
    "dataset": rest["dataset"], "query_id": rest["query_id"], "query": rest["query"],
    # l1-decided rows keep their original label; unreached rows the cascade never
    # measured (supplemented / skipped lanes / blank) get an explicit unmeasured
    # layer and NO route — an undecided row must not masquerade as a label
    "label_layer": pd.Series("l1", index=rest.index).where(
        differ, "unmeasured_" + rest["shape"].str.removeprefix("all_")),
    "route": rest["route"].where(differ, None),
    # rest rows are pure l1: final == l1; argmax / top@100 were never run for them
    **{f"score_{rt}": rest[f"score_{rt}"] for rt in ROUTES},
    **{f"score_{rt}__l1": rest[f"score_{rt}"] for rt in ROUTES},
    **{f"score_{rt}__argmax": float("nan") for rt in ROUTES},
    **{f"score_{rt}__d100": float("nan") for rt in ROUTES},
})
full = pd.concat([cascade, rest_out], ignore_index=True)
assert len(full) == len(L1), (len(full), len(L1))
assert not full.duplicated(["dataset", "query_id"]).any()
n_text = (full["query"].fillna("").str.strip() != "").sum()
full.to_parquet(CASCADE_PATH, index=False)
print(f"recorded {len(full):,} rows (all of v2-100K), query text on {n_text:,} "
      f"(missing only the {len(full) - n_text} blank-query rows) -> {CASCADE_PATH}\n")

# ------- the good-stuff ledger -------
N = len(L1)
lc = full["label_layer"].value_counts()
ledger = pd.DataFrame([
    ("l1 (existing routes_differ labels)", int(lc.get("l1", 0))),
    ("argmax(l1,l2) decisive", int(lc.get("argmax_l1_l2", 0))),
    ("depth-100 fallback decisive", int(lc.get("depth100_fallback", 0))),
    ("judge queue: tied", int(lc.get("judge_queue_tied", 0))),
    ("judge queue: zero (deep miss)", int(lc.get("judge_queue_zero", 0))),
    ("unmeasured: tied", int(lc.get("unmeasured_tied", 0))),
    ("unmeasured: zero", int(lc.get("unmeasured_zero", 0))),
], columns=["bucket", "rows"])
ledger["% of v2-100K"] = (ledger["rows"] / N * 100).round(1)
labelled = int(full["route"].notna().sum())
print(f"LABELLED (route-decided): {labelled:,} rows = {labelled / N:.1%} of v2-100K "
      f"(was {int(lc.get('l1', 0)):,} = {lc.get('l1', 0) / N:.1%} before the cascade)\n")
display(ledger)

print("\nroute mix by deciding layer (the depth layer surfaces ~3x more sparse):")
display(full[full["route"].notna()]
        .groupby("label_layer")["route"].value_counts(normalize=True)
        .mul(100).round(1).unstack(fill_value=0))

recorded 91,093 rows (all of v2-100K), query text on 91,080 (missing only the 13 blank-query rows) -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/legb_pilot/os_distill_relabel/cascade_labels.parquet

LABELLED (route-decided): 56,047 rows = 61.5% of v2-100K (was 42,871 = 47.1% before the cascade)



,bucket,rows,% of v2-100K
0,l1 (existing routes_differ labels),42871,47.1
1,"argmax(l1,l2) decisive",8132,8.9
2,depth-100 fallback decisive,5044,5.5
3,judge queue: tied,17252,18.9
4,judge queue: zero (deep miss),6728,7.4
5,unmeasured: tied,8098,8.9
6,unmeasured: zero,2968,3.3



route mix by deciding layer (the depth layer surfaces ~3x more sparse):


route,dense_only,pure_rrf,sparse_only
label_layer,,,
argmax_l1_l2,86.8,1.9,11.3
depth100_fallback,66.0,2.3,31.7
l1,47.8,5.6,46.6


In [17]:
cascade

,dataset,query_id,query,label_layer,route,score_dense_only,score_pure_rrf,score_sparse_only,score_dense_only__l1,score_pure_rrf__l1,score_sparse_only__l1,score_dense_only__argmax,score_pure_rrf__argmax,score_sparse_only__argmax,score_dense_only__d100,score_pure_rrf__d100,score_sparse_only__d100
0,clerc,100288,§ 2L1.2. The Government asserts that Rosales w...,judge_queue_tied,NaN,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,NaN,NaN,NaN
1,clerc,101226,"415 U.S. 989, 94 S.Ct. 1586, 39 L.Ed.2d 885 (1...",judge_queue_tied,NaN,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,NaN,NaN,NaN
2,clerc,101912,that exists independently of the breached cont...,argmax_l1_l2,dense_only,0.106862,0.090309,0.0,0.0,0.0,0.0,0.106862,0.090309,0.0,NaN,NaN,NaN
3,clerc,102197,1631. Section 1631 provides that a district co...,judge_queue_tied,NaN,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,NaN,NaN,NaN
4,clerc,103920,"at Exh. 1-2, 6, 8-13, 16], As a result, the Co...",argmax_l1_l2,sparse_only,0.189279,0.189279,1.0,1.0,1.0,1.0,0.189279,0.189279,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37151,webfaq-eng,993166,What are ECN Fees?,judge_queue_tied,NaN,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,NaN,NaN,NaN
37152,webfaq-eng,998175,Was Solomon the 666th person named in the Tanakh?,judge_queue_tied,NaN,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,NaN,NaN,NaN
37153,webfaq-eng,998360,What are dental alloy classifications?,judge_queue_tied,NaN,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,NaN,NaN,NaN
37154,webfaq-eng,998551,Where is Monex located and what are their busi...,judge_queue_tied,NaN,1.000000,1.000000,1.0,1.0,1.0,1.0,1.000000,1.000000,1.0,NaN,NaN,NaN


## Reading it

- **rescue_pct per lane** is the headline: what the os_distill l2 leg buys on the rows
  the v2-100K labelling could not decide. Compare against the enrichment pilot's §4
  (gemini+bm25 at arch5k scale) — the delta between the two IS the sparse-leg upgrade.
- **The dense-skew table decides the roadmap item**: if the rescued-row winner mix is
  still ~87% dense with the strong sparse leg, the skew is real signal about these
  rows, not a bm25 artifact — and residual scaling stays parked. A meaningful sparse
  share appearing here is the outcome the bake-off predicted.
- **What this does not prove**: rescued labels are scored against the same qrels as
  l1, so a rescue is internal-consistency + tie-breaking, not external router quality
  (that lives in arch5k §17–§20 and needs held-out truth). all_tied rows at the qrels
  ceiling are expected to stay tied — a high still_tied count there is the ceiling,
  not an l2 failure.
- **Skipped lanes** (§1 list) hold ~6% of the unreached population; whether any is
  worth its indexing cost is `apply_os_distill_sparse.py --plan`'s call, per the
  rows-per-dollar criteria in the cost assessment.